# Pipeline & ML Models Development

At this stage, I have a dataset ready for the development of ML models using different ML algorithms. The purpose of this is to determine which algorithms would work best with our dataset and see how I can improve on the implemented models from them.

I chose these algorithms to compare different model families: Logistic Regression (linear baseline), KNN (distance‑based), Random Forest and Gradient Boosting (tree ensembles), and SVM (margin‑based). This lets us see which learning approach best fits the dataset.

Because the dataset has numerical and categorical features with likely nonlinear patterns, tree‑based models can capture interactions well. Logistic Regression gives a simple baseline, while KNN and SVM test similarity and margin‑based learning.

In [ ]:
# Required Libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    classification_report,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.inspection import permutation_importance

from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline


# Baseline Models

In [ ]:

# dataset
df = pd.read_csv('df_cleaned.csv')

# A list of all the numerical features within the dataset
numeric_features = [
    'customer_pay',
    'warranty_pay',
    'average_cost',
    'mileage',
    'distance',
    'vehicle_age_at_service',
    'days_since_purchase',
    'days_since_last_service',
    'service_number',
    'distance_loyalty_interaction',
    'service_year',
    'service_month',
    'service_dayofweek',
    'appointment',
    'is_weekend',
    'loyalty_binary'
]

# A list of all the categorical features within the dataset
categorical_features = [
    'dealer_name',
    'make_clean'
]

# Splitting dataset into features and target
X = df.drop('next_return', axis=1)
y = df['next_return']

# Splitting the data into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Preprocessing for the pipeline: Encoding 'feature2' and scaling numerical data
preprocessor = ColumnTransformer(
    transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# List of pipelines for different models
pipelines = {
    'Logistic Regression': Pipeline(steps=[('preprocessor', preprocessor),
                                           ('classifier', LogisticRegression())]),

    'Random Forest': Pipeline(steps=[('preprocessor', preprocessor),
                                     ('classifier', RandomForestClassifier(random_state=42))]),

    'KNN': Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', KNeighborsClassifier())]),

    'SVM': Pipeline(steps=[('preprocessor', preprocessor),
                           ('classifier', SVC())]),

    'Gradient Boosting': Pipeline(steps=[('preprocessor', preprocessor),
                                         ('classifier', GradientBoostingClassifier(random_state=42))])
}

# Dictionary to store model accuracies
accuracies = {}

# Training and testing each model in the pipelines
for model_name, pipeline in pipelines.items():
    pipeline.fit(X_train, y_train)  # Fit the pipeline
    y_pred = pipeline.predict(X_test)  # Predict on test data
    accuracy = accuracy_score(y_test, y_pred)  # Calculate accuracy
    accuracies[model_name] = accuracy * 100  # Store accuracy

# Display accuracies
for model, acc in accuracies.items():
    print(f"{model} Accuracy: {acc:.2f}%")


Logistic Regression Accuracy: 79.74%
Random Forest Accuracy: 81.07%
KNN Accuracy: 77.65%
SVM Accuracy: 80.63%
Gradient Boosting Accuracy: 81.00%


**Random Forest** performed well because:

- It handles nonlinear relationships

- It captures feature interactions automatically

- It is robust to noise and outliers

In the dataset, variables like:

  - service_year
  - vehicle_age_at_service
  - days_since_last_service
  - mileage

likely interact in complex ways, which tree models capture well.

**Gradient Boosting** performs well because:

- It sequentially improves errors from previous trees

- It focuses on hard-to-classify samples

- It often achieves high predictive accuracy

So it refines patterns Random Forest might miss.

**Support Vector Machine (SVM)** performs well because:

- It works well in high-dimensional feature spaces

- The dataset likely has many encoded features

- It maximizes the margin between classes

So it can create strong separation boundaries.

**Why Logistic Regression and KNN Did Worse**

**Logistic Regression** assumes linear relationships and the data likely contains nonlinear interactions

**KNN** is sensitive to feature scaling and dimensionality. Distance-based models struggle when there are many features. This is called the curse of dimensionality.

# Data Balancing

Since the target variable was imbalanced, with fewer observations in the class representing customers who did not return, a data balancing step was necessary during model development. This was done to reduce bias toward the majority class and to improve the model’s ability to learn the patterns associated with non-returning customers. SMOTE was selected because, instead of simply repeating existing minority class cases, it creates synthetic samples based on the feature space of that class. This made it a suitable choice for improving the representation of the minority class while supporting better model learning and classification performance.

In [ ]:
# -------------------------
# Load + split
# -------------------------
df_copy = pd.read_csv('df_cleaned.csv')

X = df_copy.drop(columns=["next_return"])
y = df_copy["next_return"]

numeric_features = [
    'customer_pay','warranty_pay','average_cost','mileage','distance',
    'vehicle_age_at_service','days_since_purchase','days_since_last_service',
    'service_number','distance_loyalty_interaction','service_year',
    'service_month','service_dayofweek','appointment','is_weekend',
    'loyalty_binary'
]

categorical_features = ['dealer_name','make_clean']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# Preprocessor
# -------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

# -------------------------
# Pipelines (SMOTE happens AFTER encoding/scaling)
# -------------------------
pipelines = {
    'Random Forest': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', RandomForestClassifier(random_state=42))
    ]),
    'SVM': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', SVC())
    ]),
    'Gradient Boosting': ImbPipeline(steps=[
        ('preprocessor', preprocessor),
        ('smote', SMOTE(random_state=42)),
        ('classifier', GradientBoostingClassifier(random_state=42))
    ])
}

accuracies = {}

for model_name, pipe in pipelines.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"\n==== {model_name} ====")
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))


==== Random Forest ====
              precision    recall  f1-score   support

           0       0.59      0.39      0.47     10810
           1       0.84      0.92      0.88     37749

    accuracy                           0.80     48559
   macro avg       0.72      0.66      0.67     48559
weighted avg       0.78      0.80      0.79     48559

[[ 4212  6598]
 [ 2923 34826]]

==== SVM ====
              precision    recall  f1-score   support

           0       0.42      0.64      0.51     10810
           1       0.88      0.75      0.81     37749

    accuracy                           0.72     48559
   macro avg       0.65      0.69      0.66     48559
weighted avg       0.78      0.72      0.74     48559

[[ 6881  3929]
 [ 9523 28226]]

==== Gradient Boosting ====
              precision    recall  f1-score   support

           0       0.53      0.41      0.46     10810
           1       0.84      0.90      0.87     37749

    accuracy                           0.79     485

Random Forest achieved the best overall performance

It provided the highest accuracy and F1 score

Tree-based models performed better than SVM for this dataset

# Hyperparameter Optimization

As a standard approach to ML Development, hyperparameter Optimization is performed to see if the different metrics of the performance of the models could be improved.

In [ ]:
# A function to evaluate model performance
def evaluate(pipe, X_test, y_test, name="Model"):
    y_pred = pipe.predict(X_test)
    print(f"\n==== {name} ====")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("F1:", round(f1_score(y_test, y_pred), 4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("\nReport:\n", classification_report(y_test, y_pred))

# A pipeline function
def make_pipe(model):
    return ImbPipeline(steps=[
        ("preprocessor", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("classifier", model)
    ])

# -------------------------
# Load + split
# -------------------------
df_copy = pd.read_csv('df_cleaned.csv')

X = df_copy.drop(columns=["next_return"])
y = df_copy["next_return"]

numeric_features = [
    'customer_pay','warranty_pay','average_cost','mileage','distance',
    'vehicle_age_at_service','days_since_purchase','days_since_last_service',
    'service_number','distance_loyalty_interaction','service_year',
    'service_month','service_dayofweek','appointment','is_weekend',
    'loyalty_binary'
]

categorical_features = ['dealer_name','make_clean']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -------------------------
# Preprocessor
# -------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)


# Gradient Boosting Baseline before tuning
gb_base = make_pipe(GradientBoostingClassifier(random_state=42))
gb_base.fit(X_train, y_train)
evaluate(gb_base, X_test, y_test, "Gradient Boosting (baseline)")

# Gradient Boosting GridSearch Tuning
gb_params = {
    "classifier__n_estimators": [200, 400],
    "classifier__learning_rate": [0.05, 0.1],
    "classifier__max_depth": [2, 3],
    "classifier__subsample": [0.8, 1.0]
}

gb_search = GridSearchCV(
    estimator=gb_base,
    param_grid=gb_params,
    cv=5,
    scoring="f1",
    n_jobs=-1,
    verbose=1
)

gb_search.fit(X_train, y_train)

best_gb = gb_search.best_estimator_
print("Best GB params:", gb_search.best_params_)

evaluate(best_gb, X_test, y_test, "Gradient Boosting (tuned)")


==== Gradient Boosting (baseline) ====
Accuracy: 0.7874
F1: 0.8676
Confusion Matrix:
 [[ 4400  6410]
 [ 3913 33836]]

Report:
               precision    recall  f1-score   support

           0       0.53      0.41      0.46     10810
           1       0.84      0.90      0.87     37749

    accuracy                           0.79     48559
   macro avg       0.69      0.65      0.66     48559
weighted avg       0.77      0.79      0.78     48559

Fitting 5 folds for each of 16 candidates, totalling 80 fits


-- Incompete output!! Process is highly cpu-intensive

# Leakage-safe evaluation (Group Split by VIN)

The results look promising. It is then our perogative to confirm that the model is not learning from a form of data leakage. To check this we perform a Leakage-safe evaluation by Group split by VIN, where we ensure that vins being used in the training set are no also used in the test set

In [ ]:
# ---- Load data
df = pd.read_csv("df_cleaned_with_vin.csv")

# ---- Define target + groups (VIN must exist in the file; if not, load df before you dropped it)
y = df["next_return"]
groups = df["vin"]  # IMPORTANT: leakage-safe split based on VIN
X = df.drop(columns=["next_return"])

# ---- Feature lists
numeric_features = [
    'customer_pay','warranty_pay','average_cost','mileage','distance',
    'vehicle_age_at_service','days_since_purchase','days_since_last_service',
    'service_number','distance_loyalty_interaction','service_year',
    'service_month','service_dayofweek','appointment','is_weekend',
    'loyalty_binary'
]
categorical_features = ['dealer_name','make_clean']

# ---- Preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# ---- Leakage-safe split: VINs in train are not in test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

def evaluate(name, pipe):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    print(f"\n==== {name} ====")
    print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
    print("F1:", round(f1_score(y_test, y_pred), 4))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred))

# ---- Models (SMOTE inside pipeline)
rf_pipe = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("clf", RandomForestClassifier(random_state=42, n_jobs=-1))
])

gb_pipe = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("clf", GradientBoostingClassifier(random_state=42))
])

svm_pipe = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("clf", SVC(class_weight="balanced"))  # no predict_proba unless probability=True (kept fast)
])

evaluate("Random Forest (GroupSplit)", rf_pipe)
evaluate("Gradient Boosting (GroupSplit)", gb_pipe)
evaluate("SVM (GroupSplit)", svm_pipe)


==== Random Forest (GroupSplit) ====
Accuracy: 0.8326
F1: 0.8955
Confusion Matrix:
 [[ 5585  5225]
 [ 2892 34797]]
              precision    recall  f1-score   support

           0       0.66      0.52      0.58     10810
           1       0.87      0.92      0.90     37689

    accuracy                           0.83     48499
   macro avg       0.76      0.72      0.74     48499
weighted avg       0.82      0.83      0.83     48499


==== Gradient Boosting (GroupSplit) ====
Accuracy: 0.8273
F1: 0.892
Confusion Matrix:
 [[ 5523  5287]
 [ 3090 34599]]
              precision    recall  f1-score   support

           0       0.64      0.51      0.57     10810
           1       0.87      0.92      0.89     37689

    accuracy                           0.83     48499
   macro avg       0.75      0.71      0.73     48499
weighted avg       0.82      0.83      0.82     48499


==== SVM (GroupSplit) ====
Accuracy: 0.791
F1: 0.8596
Confusion Matrix:
 [[ 7318  3492]
 [ 6646 31043]]
      

The performance of the model nearly remained the same. This signifies that the model is learning service behavior rather than memorizing vehicles. If performance had dropped, then it would have been safe to say the model was learning information about the object identifier of the dataset

#Threshold optimization

Threshold optimization was carried out to improve the model’s classification performance beyond the default probability cutoff of 0.50. Although many classification models automatically use 0.50 as the decision threshold, this value is not always the most suitable when working with an imbalanced target variable or when the minority class is of greater interest. In this project, threshold optimization was used to identify a cutoff that provided a better balance between precision and recall for the non-returning customer class. This helped ensure that the model was not only making predictions, but was doing so in a way that better aligned with the objective of identifying customers who were less likely to return.

In [ ]:
# ---- Load data
df = pd.read_csv("df_cleaned.csv")

y = df["next_return"]
X = df.drop(columns=["next_return"])

numeric_features = [
    'customer_pay','warranty_pay','average_cost','mileage','distance',
    'vehicle_age_at_service','days_since_purchase','days_since_last_service',
    'service_number','distance_loyalty_interaction','service_year',
    'service_month','service_dayofweek','appointment','is_weekend',
    'loyalty_binary'
]
categorical_features = ['dealer_name','make_clean']

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# ---- Split (simple stratified split; teammate A covers leakage-safe group split)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Pick ONE best model (Random Forest example) + probability=True is default for RF
# Put your tuned params here if you have them:
rf = RandomForestClassifier()

pipe = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("clf", rf)
])

pipe.fit(X_train, y_train)

# ---- Probabilities for class 1 (return)
probs = pipe.predict_proba(X_test)[:, 1]

# ---- Try thresholds
thresholds = np.arange(0.30, 0.81, 0.05)

best = {"thr": None, "f1": -1}

print("thr | precision | recall | f1")
for t in thresholds:
    preds = (probs >= t).astype(int)
    p = precision_score(y_test, preds)
    r = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    print(f"{t:.2f} | {p:.3f}     | {r:.3f}  | {f1:.3f}")
    if f1 > best["f1"]:
        best = {"thr": t, "f1": f1}

print("\nBest threshold by F1:", best)

# ---- Report at best threshold
best_preds = (probs >= best["thr"]).astype(int)
print("\n==== Report at best threshold ====")
print(classification_report(y_test, best_preds))
print("Confusion Matrix:\n", confusion_matrix(y_test, best_preds))

thr | precision | recall | f1
0.30 | 0.808     | 0.983  | 0.887
0.35 | 0.815     | 0.974  | 0.888
0.40 | 0.823     | 0.963  | 0.887
0.45 | 0.831     | 0.947  | 0.885
0.50 | 0.840     | 0.928  | 0.882
0.55 | 0.848     | 0.900  | 0.873
0.60 | 0.857     | 0.861  | 0.859
0.65 | 0.868     | 0.807  | 0.836
0.70 | 0.881     | 0.731  | 0.799
0.75 | 0.893     | 0.630  | 0.739
0.80 | 0.910     | 0.499  | 0.644

Best threshold by F1: {'thr': np.float64(0.35), 'f1': 0.8878563325206981}

==== Report at best threshold ====
              precision    recall  f1-score   support

           0       0.72      0.23      0.35     10810
           1       0.82      0.97      0.89     37749

    accuracy                           0.81     48559
   macro avg       0.77      0.60      0.62     48559
weighted avg       0.79      0.81      0.77     48559

Confusion Matrix:
 [[ 2484  8326]
 [  966 36783]]


# BLOCK 3: Feature importance (Permutation Importance) + top drivers

Feature importance analysis was carried out to identify which variables contributed most to the model’s predictions. This step was useful for understanding the main factors influencing customer return behaviour and for improving the interpretability of the model. Rather than treating the model as a black box, feature importance made it possible to see which features had the strongest effect on the prediction outcome. This provided more insight into the model’s decision-making process and also helped confirm whether the selected features were meaningful in relation to the business problem.

In [ ]:
# ---- Load data
df = pd.read_csv("df_cleaned.csv")

y = df["next_return"]
X = df.drop(columns=["next_return"])

numeric_features = [
    'customer_pay','warranty_pay','average_cost','mileage','distance',
    'vehicle_age_at_service','days_since_purchase','days_since_last_service',
    'service_number','distance_loyalty_interaction','service_year',
    'service_month','service_dayofweek','appointment','is_weekend',
    'loyalty_binary'
]
categorical_features = ['dealer_name','make_clean']

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# ---- Choose one strong model (GB shown; use your tuned params if you have them)
gb = GradientBoostingClassifier(
    random_state=42,
    n_estimators=400,
    learning_rate=0.05,
    max_depth=3,
    subsample=1.0
)

pipe = ImbPipeline(steps=[
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("clf", gb)
])

pipe.fit(X_train, y_train)
preds = pipe.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, preds), 4))
print("F1:", round(f1_score(y_test, preds), 4))

# ---- Permutation importance (on test set)
# This measures how much score drops when each feature is shuffled.
result = permutation_importance(
    pipe, X_test, y_test,
    scoring="f1",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

# Permutation importance returns importances for ORIGINAL columns (before OHE),
# which is perfect for explaining to business.
imp = pd.Series(result.importances_mean, index=X_test.columns).sort_values(ascending=False)

print("\nTop 10 drivers (Permutation Importance, F1 drop):")
print(imp.head(10))

# ---- Optional: quick business insight text helpers
print("\nBusiness insight prompts:")
print("- If 'distance' is high importance: customers farther away are less likely to return.")
print("- If 'loyalty_binary'/'on_make_service' are high: loyalty and brand alignment increase retention.")
print("- If 'days_since_last_service'/'service_number' are high: service frequency is a strong retention signal.")

Accuracy: 0.8048
F1: 0.8817

Top 10 drivers (Permutation Importance, F1 drop):
service_number             0.081110
service_year               0.027192
vehicle_age_at_service     0.009599
days_since_last_service    0.002867
service_month              0.001881
appointment                0.001649
make_clean                 0.001263
average_cost               0.001163
distance                   0.000830
dealer_name                0.000448
dtype: float64

Business insight prompts:
- If 'distance' is high importance: customers farther away are less likely to return.
- If 'loyalty_binary'/'on_make_service' are high: loyalty and brand alignment increase retention.
- If 'days_since_last_service'/'service_number' are high: service frequency is a strong retention signal.
